# ROSP-23: Inter-visit interval breakdown

Diagnostic notebook for PTH Tier 1 metric **ROSP-10** (open-shutter fraction and slew/settle time), developed under ticket [ROSP-23](https://rubinobs.atlassian.net/browse/ROSP-23). Raised at the PTH steward kickoff, October 2026.

- Jira epic: [ROSP-10](https://rubinobs.atlassian.net/browse/ROSP-10)
- Jira ticket: [ROSP-23](https://rubinobs.atlassian.net/browse/ROSP-23)
- Performance Tracking Hub: [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696)

| | |
|---|---|
| Steward | Erik Dennihy |
| Owner | Sandrine Thomas |
| Category | 1 - Observatory Operations |
| Parent metric | ROSP-10 (Layer 3) |

Method version: v0.1 (draft, single-night demonstration)

Status: first implementation. This is a **local diagnostic notebook**: it has no Times Square sidecar and is not published. Changes to the method go through the PTH Method Change Process: https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/2032959501

## What this notebook measures

> For each pair of consecutive survey visits, where does the time between the end of one exposure and the start of the next actually go?

The **inter-visit interval** (the *visit gap*) is defined by OSS-REQ-0289 as the time from the shutter close of one visit to the shutter open of the next. The requirement text speaks of "second-exposure shutter close to next-visit first-exposure shutter open" — that wording predates the decision to stop taking snap pairs. **Every visit is now a single exposure**, so the interval here is simply the time between the camera's own shutter events in the EFD:

$$\text{visit gap} = t_{\texttt{startShutterOpen}}(\text{visit } N{+}1) - t_{\texttt{endShutterClose}}(\text{visit } N)$$

Every time in the analysis is therefore an EFD timestamp; the ConsDB visit table is used only to say *which* images are survey visits.

### The mechanism segments

The gap is cut into **thirteen serial segments** that sum exactly to it. Each has **one mechanism and one owner** — the team that could act on it. In time order (exact boundaries and code references in [Computation](#computation)):

| # | Segment | In one line | Owner |
|---|---|---|---|
| 1 | Script startup | script N ends, script N+1 starts running | ScriptQueue |
| 2 | Script & TCS pre-slew checks | feasibility and MTCS entry checks | Scripts / MTCS |
| 3 | Telescope prep (M1M3) | booster valves open before the slew | M1M3 |
| 4 | AOS Correction (pre-slew) | applying a correction that was already waiting | AOS |
| 5 | Slew start latency | slew command until the first axis moves | TMA / pointing |
| 6 | Mount slew+settle | axis motion and settling until both axes are in position (sub-category: limited by azimuth or elevation) | TMA |
| 7 | Rotator catch-up | rotator time beyond the mount | Rotator |
| 8 | Dome catch-up | dome time beyond mount + rotator | Dome |
| 9 | In-position checks | MTCS confirming everything is in position | OCS / MTCS |
| 10 | AOS Correction (post-slew) | applying a correction that arrived after the slew started | AOS |
| 11 | AOS pointing correction | re-pointing for the optics change of that correction | AOS / TMA |
| 12 | Camera dispatch | handing the exposure to the camera (after a post-slew correction) | OCS / Camera |
| 13 | Camera start | camera executing `takeImages` until the shutter opens | Camera / CCS |

Segments 4 and 10–12 are zero when there was no AOS correction; 7–8 are usually zero. A *filter change* is not a segment: those gaps (two slews plus the exchange) are reported separately.

## How the notebook is organised

1. **Data sources:** the visit table and the EFD topics.
2. **Selection:** which visit pairs count. Survey visits of every tier (main survey, deep drilling, twilight), including strategy changes between them — all tagged so they can be split out later; filter changes set aside; faults and interruptions removed.
3. **Computation:** the segments and their markers in one table, with code references, and the *waterfall* decomposition that sums exactly to the gap.
4. **Deep dive:** every event inside example gaps (short/typical/long, plus one with a post-slew AOS correction), laid out on timelines, so the method can be checked by eye.
5. **Full-night statistics:** per-segment time per visit — median, mean, min and max.
6. **Results:** headline box and ranked list of gap-time contributors, with a first-pass extrapolation to nights-per-year cost.
7. **Configuration-change markers:** method decisions that would break the series if changed.

In [ ]:
# Parameters cell. This notebook runs locally only (no Times Square sidecar).
day_obs = 20260713  # the night to analyse (dayObs, YYYYMMDD)
verbose_output = (
    False  # True: also show the diagnostic prints and tables (event counts, marker
)
# completeness, validation checks, selection tables, deep-dive event dumps)

In [ ]:
# Validate the parameters. vprint / vdisplay show diagnostic output only when verbose_output is True;
# the main results (statistics, figures, headline, ranking) are always shown.
from datetime import datetime

from IPython.display import display

day_obs = int(day_obs)
day_obs_date = datetime.strptime(str(day_obs), "%Y%m%d").date()
verbose_output = bool(verbose_output)


def vprint(*args, **kwargs):
    """print(), only when verbose_output is True."""
    if verbose_output:
        print(*args, **kwargs)


def vdisplay(*objs):
    """display(), only when verbose_output is True."""
    if verbose_output:
        display(*objs)


vprint(
    f"dayObs: {day_obs} ({day_obs_date.isoformat()}); verbose_output={verbose_output}"
)

## Data sources

Two kinds of input are needed: the **visit table** (which images are survey visits) and the **EFD event record** (when each exposure started and ended, and what the observatory was doing in between).

| Input | Where it comes from |
|---|---|
| **Visits** | The ConsDB, joined to FBS targets, via `rubin_nights.targets_and_visits` — the same query the nightly `FBS_Targets` Times Square notebook uses. Used for visit **identity and classification** only (science program, observation reason, band). Its `obs_start` / `obs_end` (TAI) serve only to pair each image with its shutter events. |
| **EFD events** | `lsst.sal.*` topics queried through `rubin_nights.connections.get_clients()`. EFD timestamps are UTC. **All times in the analysis come from here**, including the gap endpoints (`MTCamera.logevent_endShutterClose` / `startShutterOpen`). |

### The EFD topics used

The topic behind every segment boundary, with a link to the code that produces or waits for it, is in the segment table in [Computation](#computation). The other topics used are context only:

| Use | EFD topic |
|---|---|
| Filter-change exchange time (set-aside gaps) | `lsst.sal.MTCamera.logevent_startSetFilter` / `logevent_endSetFilter` |
| Deep-dive context | `ScriptQueue.logevent_script`, `MTAOS.logevent_closedLoopState` (all states), `MTPtg.command_poriginOffset`, `MTHexapod.logevent_inPosition`, `MTDome.logevent_azMotion`, `MTCamera.logevent_ccsCommandState`, `endReadout`, `startIntegration` |
| Observatory state (selection) | `rubin_nights.observatory_status.get_observatory_state_times` (as ROSP-09) |

In [ ]:
# Imports, and the connection to the EFD and ConsDB.
import os
import logging
import time as _time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.time import Time, TimeDelta

from rubin_nights import connections
import rubin_nights.dayobs_utils as rn_dayobs
from rubin_nights.targets_and_visits import targets_and_visits

# The Sun-position code needs the shared rubin_sim data. If it is not set, point at the shared copy at USDF.
if os.getenv("RUBIN_SIM_DATA_DIR") is None:
    for _data_dir in (
        "/sdf/data/rubin/shared/rubin_sim_data",
        "/sdf/group/rubin/shared/rubin_sim_data",
    ):
        if os.path.isdir(_data_dir):
            os.environ["RUBIN_SIM_DATA_DIR"] = _data_dir
            break

# Quieten the per-query INFO logging; failures still raise.
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("rubin_nights").setLevel(logging.WARNING)

# On an RSP the default endpoints and tokens are right; elsewhere get_clients() looks for
# your own USDF token (see rubin_nights.get_access_token). The credentials services behind
# get_clients() occasionally time out, so retry a few times before giving up.
for _attempt in range(4):
    try:
        endpoints = connections.get_clients()
        break
    except Exception as e:
        if _attempt == 3:
            raise
        print(f"get_clients() failed ({type(e).__name__}: {e}); retrying in 5 s")
        _time.sleep(5)
efd = endpoints["efd"]

In [ ]:
# The night window: the full dayObs (12:00 UTC to 12:00 UTC) for the queries, and the
# -12 deg observable night (same definition as ROSP-22 / ROSP-09) for context and for the
# nights-per-year extrapolation at the end.
TWILIGHT_ALT = -12.0  # Sun altitude (deg) bounding the observable night

t_start = rn_dayobs.day_obs_to_time(day_obs)
t_end = t_start + TimeDelta(1, format="jd")

sunset12, sunrise12 = rn_dayobs.day_obs_sunset_sunrise(day_obs, TWILIGHT_ALT)
night_hours = (sunrise12 - sunset12).jd * 24

vprint(f"dayObs window (UTC): {t_start.iso} to {t_end.iso}")
vprint(
    f"Observable night (Sun {TWILIGHT_ALT:g} deg): {sunset12.utc.iso} to {sunrise12.utc.iso} ({night_hours:.2f} h)"
)

## Selection

Selection decides **which gaps count**. A gap measures routine inter-visit overhead when both of its endpoints are survey visits, the telescope did not change filter, and the observatory was operating and healthy throughout. Every gap carries a `gap_class` label, so any class can be split out later; only faults and interruptions are removed outright. It is built in four explicit steps:

1. **Survey visits only.** Visits in the main science programs (the same `science_program` blocks and `observation_reason` exclusions as the nightly `FBS_Targets` notebook: no `field_survey_science`, no `close_loop` AOS visits). Calibrations, engineering blocks and acquisition images are out.
2. **Strata (survey tiers).** Every survey visit is tagged with its cadence type — `main_survey` (WFD/pairs/singles), `deep_drilling` (`observation_reason` starting with `ddf`) and `twilight` (the near-Sun twilight sequence). **All tiers in `SAMPLE_STRATA` are in the analysis sample** (by default, all three); the statistics also show the split by stratum, so the deep-drilling gaps can be separated when wanted.
3. **Gap classes.** The gap in front of each survey visit is classed from the previous image (in time, over *all* images taken that night):
   - `after_non_survey` — the previous image is not a survey visit (calibration, engineering…): **set aside**, the gap measures something else.
   - `filter_change` — the band changed: **set aside** and reported separately (a filter change contains two slews and the exchange). Takes precedence over a strategy change.
   - `strategy_change` — a transition between strata (e.g. twilight → main survey, into/out of a DDF sequence): **kept in the sample**, flagged (`gap_class` and the boolean `strategy_change` column) so it can be filtered out later.
   - `clean` — same stratum, same band: **in the sample**.

   The gap itself is timed from the EFD: `endShutterClose` of the previous image to `startShutterOpen` of this one (gaps whose shutter events cannot be paired are classed `no_shutter_event` and removed). **There is no gap-length cut** — a long gap with the observatory operating throughout is real overhead and is decomposed like any other.
4. **Observatory-state guard.** From the EFD observatory-state record (the same record ROSP-09 uses):
   - **Faults:** any `FAULT` period on the timeline from the *start of visit N's exposure* to the *end of visit N+1's exposure* makes the gap `fault_state` — short recoveries can hide inside otherwise ordinary-looking gaps, and a fault during an exposure can distort the gaps around it. The gaps **immediately before and after** an affected gap are set aside too (`EXCLUDE_NEIGHBOR_GAPS`), since recovery behaviour bleeds into them.
   - **Interruptions:** any period in which the observatory was not operating (`INTERRUPTION_STATES`, by default `IDLE` and `DOWNTIME`) inside the gap itself makes it an `interruption` — the queue was paused or stopped, so the gap measures the pause, not the visit overhead.

`interruption`, `fault_state` and `no_shutter_event` gaps are **excluded from this analysis entirely**, whatever their other class. Interruptions and fault behaviour are the subject of **separate analyses** (time-loss accounting and the fault-rate metrics), so here they are only identified and removed — they appear in no statistic, plot or ranking below, only in the removal log of the Selection.

The **analysis sample** is the gaps of class `clean` or `strategy_change` (`SAMPLE_CLASSES`) in the strata `SAMPLE_STRATA`.

In [ ]:
# Fetch the night's visits (ConsDB joined to FBS targets), exactly as the FBS_Targets notebook does.
# visits covers EVERY image taken in the dayObs (including calibrations); visit_gap (seconds) is the
# time from the previous image's obs_end to this image's obs_start, whatever the previous image was.
target_visits, cols, target_obs, nextvisit_visit, visits = targets_and_visits(
    t_start, t_end, endpoints, queue_index=1
)
visits = visits.sort_values("obs_start_mjd").reset_index(drop=True)
vprint(
    f"{len(visits)} images on dayObs {day_obs}, {visits.visit_id.nunique()} unique visit ids"
)
vdisplay(
    visits.groupby(["science_program", "img_type"]).size().rename("n_images").to_frame()
)

In [ ]:
# STEP 1 + 2: survey visits only, split into strata.
# The science programs and exclusions follow the FBS_Targets notebook (LSSTCam era).
SCIENCE_PROGRAMS = [
    "BLOCK-365",
    "BLOCK-407",
    "BLOCK-408",
    "BLOCK-416",
    "BLOCK-417",
    "BLOCK-419",
    "BLOCK-421",
]

is_survey = (
    visits.science_program.isin(SCIENCE_PROGRAMS)
    & (visits.observation_reason != "field_survey_science")
    & (visits.observation_reason != "block-t538")
    & ~visits.observation_reason.str.contains("close_loop", na=False)
    & (visits.img_type == "science")
)


def stratum_of(reason):
    """Cadence stratum of a survey visit, from its observation_reason."""
    if reason.startswith("ddf"):
        return "deep_drilling"
    if reason.startswith("twilight"):
        return "twilight"
    return "main_survey"


visits["stratum"] = ""
visits.loc[is_survey, "stratum"] = visits.loc[is_survey, "observation_reason"].apply(
    stratum_of
)

vprint(f"{is_survey.sum()} survey visits of {len(visits)} images")
vdisplay(
    visits[is_survey]
    .groupby(["stratum", "observation_reason"])
    .agg(n=("visit_id", "size"), median_gap=("visit_gap", "median"))
    .round(2)
)

In [ ]:
# STEP 3: classify the gap IN FRONT of every survey visit, and time it from the EFD shutter events.
# The classes are kept on every gap so any of them can be filtered later; the analysis sample is
# the strata in SAMPLE_STRATA and the classes in SAMPLE_CLASSES (strategy changes and deep drilling
# are IN the sample, tagged so they can be split out).
SAMPLE_STRATA = [
    "main_survey",
    "deep_drilling",
    "twilight",
]  # survey tiers in the analysis sample
SAMPLE_CLASSES = ["clean", "strategy_change"]  # gap classes in the analysis sample
SHUTTER_MATCH_TOL_S = (
    1.0  # max |EFD shutter event - ConsDB time| when pairing events to images
)

gaps = visits[is_survey].copy()
# The previous image in time, over ALL images of the night (so an intervening
# calibration or engineering image breaks the pair, as it should).
prev = visits.shift(1)
gaps["prev_visit_id"] = prev.loc[gaps.index, "visit_id"]
gaps["prev_stratum"] = prev.loc[gaps.index, "stratum"]
gaps["prev_band"] = prev.loc[gaps.index, "band"]


def classify(row):
    if pd.isna(row.prev_visit_id) or row.prev_stratum == "":
        return "after_non_survey"
    if row.prev_band != row.band:
        return "filter_change"  # takes precedence: a filter change cannot be decomposed
    if row.prev_stratum != row.stratum:
        return (
            "strategy_change"  # transition between cadence strata (main/DDF/twilight)
        )
    return "clean"


gaps["gap_class"] = gaps.apply(classify, axis=1)
gaps["strategy_change"] = gaps.prev_stratum.ne(gaps.stratum) & gaps.prev_stratum.ne(
    ""
)  # kept for filtering


# Gap endpoints from the EFD: MTCamera endShutterClose (of the previous image) and
# startShutterOpen (of this one). The ConsDB times only identify WHICH shutter event belongs
# to which image (nearest event within SHUTTER_MATCH_TOL_S); every time used below is EFD.
def nearest_event(event_times, targets, tol_s):
    """Nearest event to each target time (both UTC), NaT when none is within tol_s."""
    ev = pd.DatetimeIndex(event_times).asi8
    tg = pd.DatetimeIndex(targets).asi8
    idx = np.clip(np.searchsorted(ev, tg), 1, max(len(ev) - 1, 1))
    left, right = ev[idx - 1], ev[np.minimum(idx, len(ev) - 1)]
    best = np.where(np.abs(tg - left) <= np.abs(right - tg), left, right)
    ok = np.abs(best - tg) <= tol_s * 1e9
    return (
        pd.Series(pd.to_datetime(np.where(ok, best, pd.NaT.value), utc=True))
        .mask(~ok)
        .values
    )


def tai_mjd_to_utc(mjd):
    return pd.to_datetime(
        Time(np.asarray(mjd), format="mjd", scale="tai").utc.isot, utc=True
    )


ev_shutter_close = efd.select_time_series(
    "lsst.sal.MTCamera.logevent_endShutterClose", ["private_origin"], t_start, t_end
).index.sort_values()
ev_shutter_open = efd.select_time_series(
    "lsst.sal.MTCamera.logevent_startShutterOpen", ["private_origin"], t_start, t_end
).index.sort_values()
visits["t_shutter_open"] = nearest_event(
    ev_shutter_open, tai_mjd_to_utc(visits.obs_start_mjd), SHUTTER_MATCH_TOL_S
)
visits["t_shutter_close"] = nearest_event(
    ev_shutter_close, tai_mjd_to_utc(visits.obs_end_mjd), SHUTTER_MATCH_TOL_S
)

gaps["t_close"] = pd.to_datetime(
    visits.t_shutter_close.shift(1).loc[gaps.index], utc=True
)  # previous image
gaps["t_open"] = pd.to_datetime(visits.t_shutter_open.loc[gaps.index], utc=True)
gaps["t_prev_open"] = pd.to_datetime(
    visits.t_shutter_open.shift(1).loc[gaps.index], utc=True
)
gaps["t_this_close"] = pd.to_datetime(visits.t_shutter_close.loc[gaps.index], utc=True)
gaps["visit_gap_consdb"] = gaps.visit_gap  # kept for the cross-check
gaps["visit_gap"] = (
    gaps.t_open - gaps.t_close
).dt.total_seconds()  # THE gap, from the EFD

no_shutter = gaps.t_close.isna() | gaps.t_open.isna()
gaps.loc[no_shutter, "gap_class"] = "no_shutter_event"

d_gap = (gaps.visit_gap - gaps.visit_gap_consdb).dropna()
vprint(
    f"Shutter events: {len(ev_shutter_close)} endShutterClose, {len(ev_shutter_open)} startShutterOpen; "
    f"{int(no_shutter.sum())} survey gaps without a matched pair (class no_shutter_event)"
)
vprint(
    f"EFD gap - ConsDB gap: median {d_gap.median():+.3f} s, max |diff| {d_gap.abs().max():.3f} s"
)
vprint(
    f"{len(gaps)} survey-visit gaps; class counts: {gaps.gap_class.value_counts().to_dict()}"
)
vprint("The observatory-state guard (step 4, next cell) is still to be applied.")

In [ ]:
# STEP 4: the observatory-state guard, from the EFD observatory-state record (as ROSP-09).
#  - FAULT: a FAULT period anywhere on the exposure-to-exposure timeline
#      [shutter open (visit N) ... shutter close (visit N+1)]
#    disqualifies the gap (short recoveries can hide inside ordinary-looking gaps, and a fault
#    during an exposure can distort the gaps around it); by default the gaps immediately before
#    and after it are set aside too.                                         -> class fault_state
#  - INTERRUPTION: the observatory was not operating (a state in INTERRUPTION_STATES) at some
#    point inside the gap itself [shutter close ... shutter open].          -> class interruption
# Both classes override every other class and are OUT of this analysis entirely; faults and
# interruptions belong to separate analyses (ROSP-11/12, time-loss accounting). There is no
# gap-length cut: a long gap with the observatory operating throughout is real overhead.
import warnings

from rubin_nights import observatory_status

EXCLUDE_STATES = ["FAULT"]  # states that make a gap fault_state
INTERRUPTION_STATES = ["IDLE", "DOWNTIME"]  # states that make a gap an interruption
EXCLUDE_NEIGHBOR_GAPS = True  # also set aside the gaps bordering a fault-affected one

# rubin_nights 0.18.2 concatenates empty state frames and trips a pandas FutureWarning
# (observatory_status.py:692). It is harmless here; silence just that warning for this call.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", category=FutureWarning, module="rubin_nights.observatory_status"
    )
    obs_states = observatory_status.get_observatory_state_times(t_start, t_end, efd)
vprint(
    f"Observatory states on the night: "
    f"{obs_states.groupby('type').hours.sum().round(2).to_dict() if len(obs_states) else 'none recorded'} (hours)"
)


def _utc(x):
    """Datetimes as tz-aware UTC (the state table carries naive UTC timestamps)."""
    s = pd.to_datetime(pd.Series(np.asarray(x)))
    return (
        s.dt.tz_localize("UTC") if s.dt.tz is None else s.dt.tz_convert("UTC")
    ).values


def overlaps_states(states, w0, w1):
    """True where [w0, w1] overlaps any period of the given state types."""
    periods = obs_states.query("type in @states") if len(obs_states) else obs_states
    hit = np.zeros(len(w0), dtype=bool)
    for p_start, p_end in (
        zip(_utc(periods.start), _utc(periods.end)) if len(periods) else []
    ):
        hit |= (w0 < p_end) & (w1 > p_start)
    return hit


# Fault window: shutter open of visit N to shutter close of visit N+1 (EFD; ConsDB only as a fallback
# for an unmatched exposure end, whose gap is already set aside as no_shutter_event).
w0 = _utc(
    gaps.t_prev_open.fillna(
        tai_mjd_to_utc(gaps.prev_obs_start_mjd).to_series(index=gaps.index)
    )
)
w1 = _utc(
    gaps.t_this_close.fillna(
        tai_mjd_to_utc(gaps.obs_end_mjd).to_series(index=gaps.index)
    )
)
gaps["fault_overlap"] = overlaps_states(EXCLUDE_STATES, w0, w1)
# Neighbouring gaps (previous/next in time among survey gaps; `gaps` is in time order).
gaps["fault_neighbor"] = (
    gaps.fault_overlap.shift(1, fill_value=False)
    | gaps.fault_overlap.shift(-1, fill_value=False)
) & ~gaps.fault_overlap
# Interruption window: the gap itself.
has_gap = gaps.t_close.notna() & gaps.t_open.notna()
gaps["interrupted"] = False
gaps.loc[has_gap, "interrupted"] = overlaps_states(
    INTERRUPTION_STATES, _utc(gaps.t_close[has_gap]), _utc(gaps.t_open[has_gap])
)

gaps.loc[gaps.interrupted, "gap_class"] = "interruption"
fault_affected = gaps.fault_overlap | (
    gaps.fault_neighbor if EXCLUDE_NEIGHBOR_GAPS else False
)
gaps.loc[fault_affected, "gap_class"] = (
    "fault_state"  # fault takes precedence over interruption
)

vprint(
    f"Gaps overlapping a {EXCLUDE_STATES} window: {int(gaps.fault_overlap.sum())}; "
    f"neighbouring gaps: {int(gaps.fault_neighbor.sum())} (excluded: {EXCLUDE_NEIGHBOR_GAPS})"
)
vprint(
    f"Gaps with a {INTERRUPTION_STATES} period inside them: {int(gaps.interrupted.sum())}"
)

# The selection after all four steps.
summary = (
    gaps.groupby(["stratum", "gap_class"])
    .agg(
        n=("visit_id", "size"),
        median_gap=("visit_gap", "median"),
        total_min=("visit_gap", lambda s: s.sum() / 60),
    )
    .round(2)
)
vdisplay(summary)


def analysis_sample():
    """The gaps that are decomposed and reported: SAMPLE_STRATA x SAMPLE_CLASSES."""
    return gaps[gaps.stratum.isin(SAMPLE_STRATA) & gaps.gap_class.isin(SAMPLE_CLASSES)]


sample = analysis_sample()
vprint(
    f"\nAnalysis sample after the state guard: {len(sample)} gaps "
    f"(by stratum: {sample.stratum.value_counts().to_dict()}; "
    f"strategy changes: {int(sample.strategy_change.sum())})"
)

<a id="computation"></a>
## Computation

### Segments and markers

The gap is cut at **markers** — EFD events found inside the gap window `[t_close, t_open]`. Each segment **starts where the previous one ends** (the first starts at `endShutterClose` of visit N), so the table only lists where each one ends. Every marker is a SAL command or event filtered only on enum-valued fields — never a script checkpoint string.

How to read a row: **line 1** = the event that ends the segment (and the marker name in the code); **line 2** = what fills the segment. Code links point to the `main` branch, at the function that produces or waits for the event.

| # | Segment · owner | Ends at (marker) → what fills it | Code |
|---|---|---|---|
| 1 | **Script startup**<br>ScriptQueue | first main-queue `Script.state = RUNNING` (`t_run`)<br>script N finishes; the queue starts script N+1 (already loaded during the exposure) | [Script states](https://github.com/lsst-ts/ts_xml/blob/main/python/lsst/ts/xml/enums/Script.py) (ts_xml `ScriptState`; set by ts_salobj `BaseScript`) |
| 2 | **Script & TCS pre-slew checks**<br>Scripts / MTCS | first `MTM1M3.setSlewFlag` (`t_flag_set`)<br>feasibility checks; MTCS camera-cable-wrap, trackId and hexapod-compensation checks | [`BaseTrackTargetAndTakeImage.run`](https://github.com/lsst-ts/ts_standardscripts/blob/main/python/lsst/ts/standardscripts/base_track_target_and_take_image.py) → [mtcs.py `_slew_to`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/maintel/mtcs.py) |
| 3 | **Telescope prep (M1M3)**<br>M1M3 | pre-slew `MTAOS.issueCorrection` (`t_aos_pre`), else the slew command<br>booster valves open, hardpoint corrections | [mtcs.py `m1m3_booster_valve`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/maintel/mtcs.py) |
| 4 | **AOS Correction (pre-slew)**<br>AOS | `MTPtg.raDecTarget` (`t_slew_cmd`)<br>a correction was already waiting; MTCS applies it before slewing · 0 if none | [mtcs.py `handle_aos_close_loop`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/maintel/mtcs.py) (called from `_slew_to`) |
| 5 | **Slew start latency**<br>TMA / pointing | first `MTMount` azimuth/elevation `inPosition → False` (`t_motion`)<br>slew command until the first axis moves | [base_tcs.py `slew`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/base_tcs.py) (sends `raDecTarget`) |
| 6 | **Mount slew+settle**<br>TMA | last az/el `inPosition → True` (`t_mount_done`, the later of the two axes)<br>axis motion + settling · sub-category: limited by azimuth or elevation (whichever is last) | [mtcs.py `wait_for_mtmount_inposition`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/maintel/mtcs.py) |
| 7 | **Rotator catch-up**<br>Rotator | last `MTRotator.inPosition → True` (`t_rot_done`)<br>rotator time beyond the mount · usually 0 | [mtcs.py `wait_for_rotator_inposition`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/maintel/mtcs.py) |
| 8 | **Dome catch-up**<br>Dome | last `MTDomeTrajectory.telescopeVignetted → NO` (`t_dome_ok`)<br>dome time beyond mount + rotator · usually 0 | [mtcs.py `wait_for_dome_inposition`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/maintel/mtcs.py) |
| 9 | **In-position checks**<br>OCS / MTCS | second `setSlewFlag` (`t_post_flag`), else `takeImages`<br>MTCS confirmations incl. race timeouts (no events) · also holds the ~0.1 s dispatch when there is no post-slew correction | [mtcs.py `_slew_to` / `wait_for_inposition`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/maintel/mtcs.py) |
| 10 | **AOS Correction (post-slew)**<br>AOS | `MTAOS.closedLoopState → WAITING_IMAGE` (`t_aos_applied`)<br>correction that became ready after the slew started: second slew flag, `issueCorrection`, mirror/hexapod corrections + pointing offset · 0 if none | [script `wait_mtaos_idle`](https://github.com/lsst-ts/ts_maintel_standardscripts/blob/main/python/lsst/ts/maintel/standardscripts/track_target_and_take_image_lsstcam.py) · [mtaos.py `handle_corrections`](https://github.com/lsst-ts/ts_mtaos/blob/main/python/lsst/ts/mtaos/mtaos.py) |
| 11 | **AOS pointing correction**<br>AOS / TMA | second `MTM1M3.clearSlewFlag` (`t_post_clear`)<br>telescope settling onto the `MTPtg.poriginOffset` the correction sent · 0 if none | [mtaos.py `issue_pointing_correction`](https://github.com/lsst-ts/ts_mtaos/blob/main/python/lsst/ts/mtaos/mtaos.py) · [script `wait_mtaos_idle`](https://github.com/lsst-ts/ts_maintel_standardscripts/blob/main/python/lsst/ts/maintel/standardscripts/track_target_and_take_image_lsstcam.py) |
| 12 | **Camera dispatch**<br>OCS / Camera | `MTCamera.takeImages` (`t_takeimages`)<br>MTAOS-idle check, CCS-idle wait · only after a post-slew correction, else 0 | [script `_take_data`](https://github.com/lsst-ts/ts_maintel_standardscripts/blob/main/python/lsst/ts/maintel/standardscripts/track_target_and_take_image_lsstcam.py) · [base_camera.py `expose`](https://github.com/lsst-ts/ts_observatory_control/blob/main/python/lsst/ts/observatory/control/base_camera.py) |
| 13 | **Camera start**<br>Camera / CCS | `MTCamera.startShutterOpen` (`t_open`)<br>camera executing `takeImages` until the shutter opens (~0.15 s) | [MTCamera events](https://github.com/lsst-ts/ts_xml/blob/main/python/lsst/ts/xml/data/sal_interfaces/MTCamera/MTCamera_Events.xml) (CCS camera code is not in lsst-ts) |

Rules that make the segments add up:

- **Sum to the gap.** Markers are clipped into the gap; a missing marker inherits the neighbouring boundary (its segment is 0). So `sum(segments) = gap` for every gap.
- **Marginal attribution.** Systems that move in parallel are charged only the time they add beyond the critical path: rotator and dome *beyond* the mount.
- **Mount sub-category.** Row 6 is also split by **which axis was last in position**: the whole mount time goes to that axis (`c_slew_az` / `c_slew_el`, label `slew_limited_by`), so the two add up to the segment. This split is kept for exploration in the statistics, not shown as segments. Each axis's own motion time (`az_axis_s`, `el_axis_s`) is a non-additive diagnostic.
- **Post-slew boundary.** The slew markers (rows 6–8) are searched only *before* `t_post_flag`, so the re-pointing after a post-slew correction is charged to rows 10–11.
- **First / last.** Commands and script states use the first event in the window; in-position transitions and `takeImages` use the last.

Caveats:

- *Camera dispatch* can only be separated after a post-slew correction; otherwise it sits inside *In-position checks* (~0.1–0.5 s).
- *In-position checks* and *AOS pointing correction* are mostly assume-in-position race timeouts — idle time with **no event trail**. The pointing correction is very regular (~3.0 s on the development night), consistent with a fixed timeout rather than real motion: the mount axes never report leaving position for the small offset.
- If the mount does not move at all (e.g. a small deep-drilling dither with only a rotator move), rows 5–6 are 0 and the time goes to the slower system (usually *Rotator catch-up*); the statistics list such gaps.

In [ ]:
# Query the EFD once for the whole night, one topic per marker.
# Each query returns a time-indexed DataFrame (UTC). (The shutter events bounding the gaps were
# fetched in the Selection: ev_shutter_close / ev_shutter_open.)
def efd_times(topic, query_filter=None, fields=None, index=None):
    """Sorted UTC timestamps of one EFD topic over the night (optionally filtered)."""
    d = efd.select_time_series(
        topic, fields if fields else "*", t_start, t_end, index=index
    )
    if len(d) > 0 and query_filter is not None:
        d = d.query(query_filter)
    return d.index.sort_values()


ev_flag_set = efd_times(
    "lsst.sal.MTM1M3.command_setSlewFlag", fields=["private_identity"]
)
ev_flag_clear = efd_times(
    "lsst.sal.MTM1M3.command_clearSlewFlag", fields=["private_identity"]
)
ev_aos = efd_times(
    "lsst.sal.MTAOS.command_issueCorrection", fields=["private_sndStamp"]
)
# MTAOS closedLoopState is a ts_xml enum (ClosedLoopState: IDLE=1, WAITING_IMAGE=2, PROCESSING=3,
# WAITING_APPLY=4, APPLYING_CORRECTION=5, ERROR=6). MTAOS writes WAITING_IMAGE as soon as it has
# finished applying a correction (mirror/hexapod corrections and the MTPtg pointing offset).
ev_aos_applied = efd_times(
    "lsst.sal.MTAOS.logevent_closedLoopState", "state == 2", ["state"]
)
ev_slew_cmd = efd_times("lsst.sal.MTPtg.command_raDecTarget", fields=["targetName"])
_az = efd.select_time_series(
    "lsst.sal.MTMount.logevent_azimuthInPosition", ["inPosition"], t_start, t_end
)
_el = efd.select_time_series(
    "lsst.sal.MTMount.logevent_elevationInPosition", ["inPosition"], t_start, t_end
)
ev_az_done = _az.query("inPosition == True").index.sort_values()
ev_el_done = _el.query("inPosition == True").index.sort_values()
ev_az_move = _az.query(
    "inPosition == False"
).index.sort_values()  # axis leaves position: motion starts
ev_el_move = _el.query("inPosition == False").index.sort_values()
ev_rot_done = efd_times(
    "lsst.sal.MTRotator.logevent_inPosition", "inPosition == True", ["inPosition"]
)
ev_dome_ok = efd_times(
    "lsst.sal.MTDomeTrajectory.logevent_telescopeVignetted",
    "vignetted == 1",
    ["vignetted"],
)
ev_takeimages = efd_times("lsst.sal.MTCamera.command_takeImages", fields=["numImages"])
ev_filter_start = efd_times(
    "lsst.sal.MTCamera.logevent_startSetFilter", fields=["filterName"]
)
ev_filter_end = efd_times(
    "lsst.sal.MTCamera.logevent_endSetFilter", fields=["filterName"]
)

# Script lifecycle on the MAIN queue (script salIndex 1xxxxx).
# Only the RUNNING state transition is used as a marker. The state value is a ts_xml enum (ScriptState.RUNNING == 3).
script_states = efd.select_time_series(
    "lsst.sal.Script.logevent_state", ["state", "salIndex"], t_start, t_end
)
main_q = (
    script_states.query("salIndex < 200000") if len(script_states) else script_states
)
ev_script_run = main_q.query("state == 3").index.sort_values()

for name, ev in [
    ("setSlewFlag", ev_flag_set),
    ("clearSlewFlag", ev_flag_clear),
    ("issueCorrection", ev_aos),
    ("AOS WAITING_IMAGE", ev_aos_applied),
    ("raDecTarget", ev_slew_cmd),
    ("azInPosition=False", ev_az_move),
    ("elInPosition=False", ev_el_move),
    ("azInPosition=True", ev_az_done),
    ("elInPosition=True", ev_el_done),
    ("rotInPosition=True", ev_rot_done),
    ("domeVignetted=NO", ev_dome_ok),
    ("takeImages", ev_takeimages),
    ("script RUNNING", ev_script_run),
    ("startSetFilter", ev_filter_start),
    ("endSetFilter", ev_filter_end),
]:
    vprint(f"{name:20s} {len(ev):6d} events")

In [ ]:
# Assign markers to each gap: the FIRST event of a kind inside the window for commands and
# script states (one per gap), the LAST for in-position / vignetting transitions and for
# takeImages (issued just before the exposure). Comparisons are done on int64 nanoseconds
# since epoch to avoid timezone dtype pitfalls.
def first_in_window(event_times, lo, hi):
    """First event in [lo, hi] for each window; NaT when there is none."""
    ev = pd.DatetimeIndex(event_times).asi8
    lo_ns, hi_ns = pd.DatetimeIndex(lo).asi8, pd.DatetimeIndex(hi).asi8
    idx = np.searchsorted(ev, lo_ns, side="left")
    cand = np.where(
        idx < len(ev), ev[np.minimum(idx, len(ev) - 1)], np.iinfo(np.int64).max
    )
    found = (idx < len(ev)) & (cand <= hi_ns)
    out = np.where(found, cand, pd.NaT.value)
    return pd.Series(pd.to_datetime(out, utc=True), index=lo.index).mask(~found)


def last_in_window(event_times, lo, hi):
    """Last event in [lo, hi] for each window; NaT when there is none."""
    ev = pd.DatetimeIndex(event_times).asi8
    lo_ns, hi_ns = pd.DatetimeIndex(lo).asi8, pd.DatetimeIndex(hi).asi8
    idx = np.searchsorted(ev, hi_ns, side="right") - 1
    cand = np.where(idx >= 0, ev[np.maximum(idx, 0)], np.iinfo(np.int64).min)
    found = (idx >= 0) & (cand >= lo_ns)
    out = np.where(found, cand, pd.NaT.value)
    return pd.Series(pd.to_datetime(out, utc=True), index=lo.index).mask(~found)


# Markers only for gaps with both shutter events (the others are set aside).
g_ok = gaps.t_close.notna() & gaps.t_open.notna()
lo, hi = gaps.loc[g_ok, "t_close"], gaps.loc[g_ok, "t_open"]
ONE_NS = pd.Timedelta(1, "ns")
m = pd.DataFrame(index=lo.index)
m["t_run"] = first_in_window(ev_script_run, lo, hi)  # script N+1 goes RUNNING
m["t_flag_set"] = first_in_window(ev_flag_set, lo, hi)
m["t_slew_cmd"] = first_in_window(ev_slew_cmd, lo, hi)
slew_or_open = m.t_slew_cmd.fillna(hi)
# AOS CORRECTION (PRE-SLEW): the closed loop was already WAITING_APPLY when the visit started, so
# MTCS issues (and awaits) the correction before the slew command.
m["t_aos_pre"] = first_in_window(ev_aos, lo, slew_or_open)
# AOS CORRECTION (POST-SLEW): the closed loop went WAITING_APPLY during the slew or the in-position
# checks. The script's take-data step (wait_mtaos_idle) then runs a SECOND M1M3 slew-flag cycle:
#   setSlewFlag -> issueCorrection (MTAOS applies mirror/hexapod corrections and sends the MTPtg
#   pointing offset, then writes closedLoopState WAITING_IMAGE) -> wait_for_inposition -> clearSlewFlag
# and only then dispatches the exposure. The cycle is accepted only when an issueCorrection follows
# the second setSlewFlag inside the gap.
second_flag = first_in_window(ev_flag_set, slew_or_open + ONE_NS, hi)
m["t_aos_post"] = first_in_window(ev_aos, second_flag.fillna(hi), hi)
m["t_post_flag"] = second_flag.where(m.t_aos_post.notna())
post_or_open = m.t_post_flag.fillna(hi)
m["t_aos_applied"] = first_in_window(
    ev_aos_applied, m.t_aos_post.fillna(hi), hi
)  # WAITING_IMAGE
m["t_post_clear"] = last_in_window(ev_flag_clear, post_or_open, hi).where(
    m.t_post_flag.notna()
)

# The FIRST slew's markers stop where the post-slew AOS cycle begins, so that cycle's own
# in-position toggles are charged to it and not to the mount/rotator/dome.
m["t_az_move"] = first_in_window(ev_az_move, slew_or_open, post_or_open)
m["t_el_move"] = first_in_window(ev_el_move, slew_or_open, post_or_open)
m["t_motion"] = m[["t_az_move", "t_el_move"]].min(axis=1)  # first axis leaves position
m["t_az_done"] = last_in_window(ev_az_done, lo, post_or_open)
m["t_el_done"] = last_in_window(ev_el_done, lo, post_or_open)
m["t_mount_done"] = m[["t_az_done", "t_el_done"]].max(axis=1)
m["t_rot_done"] = last_in_window(ev_rot_done, lo, post_or_open)
m["t_dome_ok"] = last_in_window(ev_dome_ok, lo, post_or_open)
m["t_takeimages"] = last_in_window(ev_takeimages, lo, hi)
for c in m.columns:
    gaps[c] = m[c].reindex(gaps.index)  # NaT for gaps without shutter events
gaps["second_flag_cycle"] = (second_flag.notna()).reindex(gaps.index, fill_value=False)

# How complete are the markers in the analysis sample?
sample = analysis_sample()
marker_cols = [
    "t_run",
    "t_flag_set",
    "t_aos_pre",
    "t_slew_cmd",
    "t_motion",
    "t_mount_done",
    "t_rot_done",
    "t_dome_ok",
    "t_post_flag",
    "t_aos_post",
    "t_aos_applied",
    "t_post_clear",
    "t_takeimages",
]
vprint(f"Marker completeness over {len(sample)} gaps in the analysis sample:")
for c in marker_cols:
    vprint(f"  {c:15s} {sample[c].notna().mean():6.1%}")
post = sample[sample.t_post_flag.notna()]
vprint(
    f"Post-slew AOS cycles: {len(post)}; with WAITING_IMAGE found: {post.t_aos_applied.notna().sum()}, "
    f"with the closing clearSlewFlag found: {post.t_post_clear.notna().sum()}"
)
# A second slew-flag cycle with no issueCorrection would be left in In-position checks; it should not
# occur in the sample (filter-change gaps have one by construction -- two slews -- but are not decomposed).
n_flag_no_aos = int((sample.second_flag_cycle & sample.t_aos_post.isna()).sum())
vprint(
    f"Second slew-flag cycles WITHOUT an issueCorrection in the sample: {n_flag_no_aos}"
)

In [ ]:
# THE DECOMPOSITION: cut each gap into serial segments between consecutive markers.
# Markers are clipped into the gap and a missing marker inherits the neighbouring boundary
# (its segment is then 0), so the segments sum exactly to the gap.
def seconds(a, b):
    """(b - a) in seconds, elementwise, NaN-safe."""
    return (b - a).dt.total_seconds()


d = gaps  # work on all gaps; statistics later select the analysis sample

# Boundary chain, in time order: t_close <= b_run <= ... <= b_take <= t_open. Every boundary is a
# SAL command or event (enum-valued where filtered) — never a checkpoint string, which belongs to
# the observing script and can change with any script release.
# b_slew is fixed before b_aos_pre so a missing AOS marker collapses the pre-slew AOS segment to
# zero and leaves the whole prep window in telescope prep (not the reverse); likewise b_take is
# fixed before the post-slew boundaries, which collapse onto it when there is no post-slew cycle.
b_run = d.t_run.fillna(d.t_close).clip(
    lower=d.t_close, upper=d.t_open
)  # script RUNNING
b_flag = d.t_flag_set.fillna(b_run).clip(
    lower=b_run, upper=d.t_open
)  # M1M3 slew flag set
b_slew = d.t_slew_cmd.fillna(b_flag).clip(lower=b_flag, upper=d.t_open)  # raDecTarget
b_aos_pre = d.t_aos_pre.fillna(b_slew).clip(
    lower=b_flag, upper=b_slew
)  # pre-slew issueCorrection
b_motion = d.t_motion.fillna(b_slew).clip(
    lower=b_slew, upper=d.t_open
)  # first axis leaves position
b_mount = d.t_mount_done.fillna(b_motion).clip(
    lower=b_motion, upper=d.t_open
)  # az+el in position
b_rot = d.t_rot_done.fillna(b_mount).clip(
    lower=b_mount, upper=d.t_open
)  # rotator BEYOND the mount
b_dome = d.t_dome_ok.fillna(b_rot).clip(
    lower=b_rot, upper=d.t_open
)  # dome beyond mount+rotator
b_take = d.t_takeimages.fillna(d.t_open).clip(
    lower=b_dome, upper=d.t_open
)  # exposure handed to camera
b_post = d.t_post_flag.fillna(b_take).clip(
    lower=b_dome, upper=b_take
)  # 2nd setSlewFlag (post-slew AOS)
b_post_clear = d.t_post_clear.fillna(b_take).clip(
    lower=b_post, upper=b_take
)  # its clearSlewFlag
b_applied = d.t_aos_applied.fillna(b_post_clear).clip(
    lower=b_post, upper=b_post_clear
)  # AOS WAITING_IMAGE

d["c_script_startup"] = seconds(
    d.t_close, b_run
)  # script N finishes (takeImages ack at end of
# acquisition) + queue hand-off -> N+1 RUNNING
d["c_preslew"] = seconds(b_run, b_flag)  # N+1 feasibility checks + MTCS pre-slew checks
# (CCW, trackId, hexapod comp) -> M1M3 slew flag
d["c_slew_prep"] = seconds(b_flag, b_aos_pre)  # M1M3 booster valve / hardpoint prep
d["c_aos_pre"] = seconds(
    b_aos_pre, b_slew
)  # pre-slew AOS correction (0 when none was pending)
d["c_slew_latency"] = seconds(
    b_slew, b_motion
)  # slew command -> first axis leaves position
d["c_slew_mount"] = seconds(
    b_motion, b_mount
)  # mount motion + settling -> both axes in position
d["c_settle"] = seconds(b_mount, b_rot)  # rotator catch-up beyond the mount
d["c_dome"] = seconds(b_rot, b_dome)  # dome catch-up beyond mount + rotator
d["c_inpos"] = seconds(b_dome, b_post)  # MTCS in-position checks (rotator/hexapod/dome
# confirmations incl. race timeouts) until the slew
# returns; without a post-slew AOS cycle this also
# holds the ~0.1 s camera dispatch (no event splits it)
d["c_aos_post"] = seconds(
    b_post, b_applied
)  # post-slew AOS correction: 2nd slew flag ->
# issueCorrection -> MTAOS done (WAITING_IMAGE)
d["c_aos_pointing"] = seconds(
    b_applied, b_post_clear
)  # AOS pointing correction: telescope back in
# position after the offset -> clearSlewFlag
d["c_dispatch"] = seconds(
    b_post_clear, b_take
)  # camera dispatch after a post-slew AOS cycle
# (MTAOS-idle check, CCS-idle wait) -> takeImages
d["c_cam_start"] = seconds(
    b_take, d.t_open
)  # camera executing takeImages -> shutter open

# SUB-CATEGORY of the mount segment (for exploration; not a segment of its own): which axis
# limits the slew -- the one whose in-position report comes last. Charging the whole motion to
# it splits c_slew_mount exactly into c_slew_az + c_slew_el (one of them is 0 in every gap).
# The per-axis motion times (NOT additive -- the axes move in parallel) are kept as diagnostics.
az_limits = d.t_az_done.notna() & ~(d.t_el_done > d.t_az_done)
d["slew_limited_by"] = np.where(
    d.t_mount_done.isna(),
    "none (mount did not move)",
    np.where(az_limits, "azimuth", "elevation"),
)
d["c_slew_az"] = d.c_slew_mount.where(az_limits, 0.0)
d["c_slew_el"] = d.c_slew_mount - d.c_slew_az
d["az_axis_s"] = seconds(b_motion, d.t_az_done.clip(lower=b_motion, upper=b_mount))
d["el_axis_s"] = seconds(b_motion, d.t_el_done.clip(lower=b_motion, upper=b_mount))

# THE THIRTEEN MECHANISM SEGMENTS, in timeline order, with the display names and colours
# used by every table and plot below. One mechanism, one owner per segment. (The v0.1 draft
# split c_preslew at the "Track target" checkpoint and the post-slew checks at the "Take data."
# checkpoint; the checkpoint strings were an unacceptable dependence on observing-script text,
# so those splits were dropped — see Configuration-change markers.)
SEGMENTS = {
    "c_script_startup": ("Script startup", "#8c6bb1"),
    "c_preslew": ("Script & TCS pre-slew checks", "#9e9ac8"),
    "c_slew_prep": ("Telescope prep (M1M3)", "#9ecae1"),
    "c_aos_pre": ("AOS Correction (pre-slew)", "#fd8d3c"),
    "c_slew_latency": ("Slew start latency", "#c6dbef"),
    "c_slew_mount": ("Mount slew+settle", "#3182bd"),
    "c_settle": ("Rotator catch-up", "#74c476"),
    "c_dome": ("Dome catch-up", "#c7a44a"),
    "c_inpos": ("In-position checks", "#66c2a5"),
    "c_aos_post": ("AOS Correction (post-slew)", "#d94801"),
    "c_aos_pointing": ("AOS pointing correction", "#fdae6b"),
    "c_dispatch": ("Camera dispatch", "#bc80bd"),
    "c_cam_start": ("Camera start", "#e377c2"),
}
SUBCOMPONENTS = list(SEGMENTS)
MECH_COMPONENTS = [name for name, _ in SEGMENTS.values()]
SEGMENT_NAME = {col: name for col, (name, _) in SEGMENTS.items()}
SEGMENT_COLOR = {name: color for _, (name, color) in SEGMENTS.items()}
for col, name in SEGMENT_NAME.items():
    d[name] = d[col]

# Validation 1: the segments must sum to the gap to numerical precision, and the mount
# sub-categories must sum to the mount segment. (The assert always runs; the numbers are
# printed only when verbose_output is True.)
sample = analysis_sample()
resid = (sample[SUBCOMPONENTS].sum(axis=1) - sample.visit_gap).abs()
vprint(
    f"max |sum(segments) - gap| over {len(sample)} gaps in the analysis sample: {resid.max():.4f} s"
)
assert resid.max() < 0.05, "segments do not sum to the gap"
resid_axes = (sample.c_slew_az + sample.c_slew_el - sample.c_slew_mount).abs().max()
vprint(f"max |azimuth + elevation sub-categories - mount segment|: {resid_axes:.4f} s")

# Validation 2: the AOS segments (and the post-cycle dispatch) must be zero when no such correction
# was issued.
no_post = sample.t_post_flag.isna()
for col, mask in [
    ("c_aos_pre", sample.t_aos_pre.isna()),
    ("c_aos_post", no_post),
    ("c_aos_pointing", no_post),
    ("c_dispatch", no_post),
]:
    leak = sample.loc[mask, col].abs().max()
    vprint(
        f"max {col} seconds charged to gaps without that correction: "
        f"{0.0 if np.isnan(leak) else leak:.4f} s"
    )

## Deep dive: example visit gaps, event by event

Before trusting statistics built from the markers, look at individual gaps in full. The next cells take the gaps in the analysis sample closest to the **10th percentile** (a short gap), the **median** (a typical gap) and the **99th percentile** (a long gap) of the night's gap distribution, plus — when the night has any — the typical gap with a **post-slew AOS correction** (the one closest to the median of those gaps), and for each:

1. print **every EFD event** inside it, in time order, with the script identity on the M1M3 flag commands (cross-checking the window assignment against the script that actually ran), and
2. draw it as a figure with two panels on a shared time axis:
   - **top — the critical path:** one horizontal bar per mechanism segment, stepping down in time order, with its duration written next to it. Dotted vertical connectors mark the hand-off from each segment to the next; together the bars tile the gap exactly from shutter close to shutter open.
   - **bottom — the raw events:** one row per subsystem, a tick per EFD event (`F`/`T` = `inPosition` False/True; labels are thinned in dense clusters — the printed dump is the complete record).

What the rows show, segment by segment:

- **Scripts / queue** (`ScriptQueue.logevent_script` + `Script.logevent_state`, labelled `NNN:state` with the script index mod 1000) bounds the **Script startup** segment: script N goes `done` within ~0.1 s of shutter close (its `takeImages` command is acked at the end of acquisition), and script N+1 — loaded and configured during the exposure — goes `run`. The script's checkpoints re-emit the `run` state during the gap; they appear in the printed dump as *content* (useful for seeing what the script is doing) but are **never used as segment boundaries** — checkpoint text can change with any script release. The queue's `nextVisit` event is *not* shown: it announces the visit after next and bounds nothing.
- **Mount / pointing** shows the slew command and both axes leaving (`az F`, `el F`: the end of the **Slew start latency**) and reaching position (`az T`, `el T`); the last `T` ends the **Mount slew+settle**, and which axis it belongs to is the slew's limiting axis (the sub-category). `ptg offset` is the pointing correction MTAOS sends with a correction.
- **AOS** shows the closed-loop state (`WAIT APPLY` = a correction is ready, `wait img` = MTAOS has finished applying one) and `issueCorrection`. A correction ready *before* the slew is applied before the slew command (**AOS Correction (pre-slew)**). One that becomes ready *later* is applied after the in-position checks, between a second `set`/`clear` pair on the **M1M3 flag** row: `set` → `issueCorrection` → `wait img` is the **AOS Correction (post-slew)**, `wait img` → `clear` is the **AOS pointing correction**, and `clear` → `takeImages` is the **Camera dispatch**.
- **Rotator / hexapods** make the **In-position checks** content visible: after the mount is in position MTCS confirms the rotator (when no event arrives, that is the assume-in-position race timeout — idle time with no event trail) and the hexapods, which toggle continuously with compensation updates (hence ticks only). The CCS command-state row shows the camera dispatch is nearly instantaneous once the script gets there.

All figures share one time axis (set by the longest gap), so bar lengths compare directly: the short gap is close to the floor everywhere, while the long gap usually earns its length in one or two specific segments rather than uniformly.

In [ ]:
# Pick the gaps in the analysis sample closest to the 10th percentile, the median and the 99th
# percentile of the gap distribution, plus (when the night has any) the typical gap with a
# POST-SLEW AOS correction, and fetch every EFD event inside each of them (for the figures below;
# the full event dumps are printed only when verbose_output is True).
sample = analysis_sample()

DEEP_DIVE_PICKS = {
    "short (10th percentile)": 0.10,
    "typical (median)": 0.50,
    "long (99th percentile)": 0.99,
}
deep_gaps = {}
for pick_label, q in DEEP_DIVE_PICKS.items():
    target = sample.visit_gap.quantile(q)
    deep_gaps[pick_label] = sample.iloc[
        (sample.visit_gap - target).abs().argsort().iloc[0]
    ]
with_post_aos = sample[sample.t_aos_post.notna()]
if len(with_post_aos) > 0:
    target = with_post_aos.visit_gap.median()
    deep_gaps["post-slew AOS correction (median of those gaps)"] = with_post_aos.iloc[
        (with_post_aos.visit_gap - target).abs().argsort().iloc[0]
    ]

# Topics chosen so every segment's CONTENT is visible, not just its boundaries:
#  - ScriptQueue.logevent_script / Script.logevent_state show what fills Script startup
#    (script N finishing, the queue hand-off, script N+1 starting and its checkpoints —
#    checkpoints are shown here as content only; they are never segment boundaries).
#    The queue's nextVisit event is deliberately NOT shown: it announces the visit AFTER
#    next and bounds nothing.
#  - Hexapod in-position and CCS command-state events show what fills the In-position checks
#    and Camera dispatch (rotator/hexapod confirmation waits, CCS busy/idle, dispatch).
#  - MTAOS closedLoopState shows when a correction becomes ready (WAITING_APPLY = 4) and when
#    MTAOS has finished applying it (WAITING_IMAGE = 2); MTPtg poriginOffset is the pointing
#    offset MTAOS sends as part of the correction.
DEEP_DIVE_TOPICS = [
    ("lsst.sal.MTCamera.logevent_endShutterClose", []),
    ("lsst.sal.MTCamera.logevent_endReadout", ["imageName"]),
    (
        "lsst.sal.ScriptQueue.logevent_script",
        ["scriptSalIndex", "processState", "scriptState"],
    ),
    ("lsst.sal.Script.logevent_state", ["state", "lastCheckpoint", "salIndex"]),
    ("lsst.sal.MTM1M3.command_setSlewFlag", ["private_identity"]),
    ("lsst.sal.MTM1M3.command_clearSlewFlag", ["private_identity"]),
    ("lsst.sal.MTAOS.command_issueCorrection", ["private_identity"]),
    ("lsst.sal.MTAOS.logevent_closedLoopState", ["state"]),
    ("lsst.sal.MTPtg.command_raDecTarget", ["targetName"]),
    ("lsst.sal.MTPtg.command_poriginOffset", ["private_identity"]),
    ("lsst.sal.MTMount.logevent_azimuthInPosition", ["inPosition"]),
    ("lsst.sal.MTMount.logevent_elevationInPosition", ["inPosition"]),
    ("lsst.sal.MTRotator.logevent_inPosition", ["inPosition"]),
    ("lsst.sal.MTHexapod.logevent_inPosition", ["inPosition", "salIndex"]),
    ("lsst.sal.MTDome.logevent_azMotion", ["inPosition"]),
    ("lsst.sal.MTDomeTrajectory.logevent_telescopeVignetted", ["vignetted"]),
    ("lsst.sal.MTCamera.logevent_ccsCommandState", ["substate"]),
    ("lsst.sal.MTCamera.command_takeImages", ["numImages"]),
    ("lsst.sal.MTCamera.logevent_startIntegration", ["imageName"]),
    ("lsst.sal.MTCamera.logevent_startShutterOpen", []),
]


def fetch_gap_events(g, pad_s=2):
    """Deep-dive EFD events in [t_close - pad, t_open + pad] of gap row `g`, time-sorted.

    Returns (timestamp, name, info-dict) with name like 'MTCamera.endShutterClose'.
    """
    t0 = Time(g.t_close.tz_localize(None)) - TimeDelta(pad_s, format="sec")
    t1 = Time(g.t_open.tz_localize(None)) + TimeDelta(pad_s, format="sec")
    events = []
    for topic, fields in DEEP_DIVE_TOPICS:
        try:
            dd = efd.select_time_series(topic, fields if fields else "*", t0, t1)
        except Exception:
            continue  # topic absent in this EFD epoch; skip rather than fail the dump
        name = topic.removeprefix("lsst.sal.").replace("logevent_", "")
        for ts, row in dd.iterrows():
            events.append((ts, name, {f: row[f] for f in fields if f in dd.columns}))
    return sorted(events, key=lambda e: e[0])


deep_events_by_pick = {}
for pick_label, g in deep_gaps.items():
    deep_events_by_pick[pick_label] = fetch_gap_events(g)
    vprint(
        f"===== {pick_label}: gap before visit {g.visit_id} "
        f"({g.visit_gap:.2f} s, {g.observation_reason}, band {g.prev_band}->{g.band}) ====="
    )
    for ts, name, info in deep_events_by_pick[pick_label]:
        dt = (ts - g.t_close).total_seconds()
        info_str = ", ".join(f"{k}={v}" for k, v in info.items())
        vprint(f"{dt:+8.2f} s  {name:34s} {info_str}")
    vprint()

# One line per pick when not verbose, so the figures below stay identifiable.
if not verbose_output:
    for pick_label, g in deep_gaps.items():
        print(
            f"{pick_label}: gap before visit {g.visit_id} ({g.visit_gap:.2f} s, {g.observation_reason})"
        )

In [ ]:
# Each deep-dive gap as two panels on a shared time axis:
#  TOP:    the critical path -- one bar per mechanism segment, stepping down in time order,
#          duration written beside/inside each bar, dotted connectors at the hand-offs.
#  BOTTOM: the raw EFD events, one row per subsystem. The scripts row shows the queue
#          hand-off that fills Script startup (NNN:done -> NNN+1:run, script indices mod
#          1000). The hexapods row is ticks-only: the hexapods toggle continuously with
#          compensation updates, so per-event labels would flood the panel.
# All the figures use ONE common time axis (set by the longest picked gap), so bar
# lengths are directly comparable across the examples.
SCRIPT_STATE = {
    0: "?",
    1: "unconf",
    2: "conf",
    3: "run",
    4: "paused",
    5: "ending",
    6: "stopping",
    7: "failing",
    8: "done",
    9: "stopped",
    10: "failed",
    11: "conf_failed",
}
PROCESS_STATE = {
    1: "loading",
    2: "configured",
    3: "running",
    4: "done",
    5: "loadfail",
    6: "conffail",
    7: "terminated",
}
AOS_LOOP_STATE = {
    1: "idle",
    2: "wait img",
    3: "processing",
    4: "WAIT APPLY",
    5: "applying",
    6: "error",
}
EVENT_LABEL = {
    "MTCamera.endShutterClose": "shutter close",
    "MTCamera.endReadout": "end readout",
    "MTCamera.command_takeImages": "takeImages",
    "MTCamera.startIntegration": "start integ",
    "MTCamera.startShutterOpen": "shutter open",
    "MTM1M3.command_setSlewFlag": "set",
    "MTM1M3.command_clearSlewFlag": "clear",
    "MTAOS.command_issueCorrection": "issueCorrection",
    "MTPtg.command_raDecTarget": "slew cmd",
    "MTPtg.command_poriginOffset": "ptg offset",
    "MTMount.azimuthInPosition": "az",
    "MTMount.elevationInPosition": "el",
    "MTRotator.inPosition": "rot",
    "MTDome.azMotion": "dome",
    "MTDomeTrajectory.telescopeVignetted": "vig",
}
# (row label, event names, colour, label_events) -- label_events=False plots ticks only.
ROW_STREAMS = [
    (
        "camera",
        [
            "MTCamera.endShutterClose",
            "MTCamera.endReadout",
            "MTCamera.ccsCommandState",
            "MTCamera.command_takeImages",
            "MTCamera.startIntegration",
            "MTCamera.startShutterOpen",
        ],
        "k",
        True,
    ),
    ("scripts / queue", ["ScriptQueue.script", "Script.state"], "#8c6bb1", True),
    (
        "M1M3 flag",
        ["MTM1M3.command_setSlewFlag", "MTM1M3.command_clearSlewFlag"],
        "#9ecae1",
        True,
    ),
    (
        "AOS",
        ["MTAOS.command_issueCorrection", "MTAOS.closedLoopState"],
        "#fd8d3c",
        True,
    ),
    (
        "mount / pointing",
        [
            "MTPtg.command_raDecTarget",
            "MTPtg.command_poriginOffset",
            "MTMount.azimuthInPosition",
            "MTMount.elevationInPosition",
        ],
        "#3182bd",
        True,
    ),
    ("rotator", ["MTRotator.inPosition"], "#74c476", True),
    ("hexapods (ticks only)", ["MTHexapod.inPosition"], "#17becf", False),
    (
        "dome",
        ["MTDome.azMotion", "MTDomeTrajectory.telescopeVignetted"],
        "#c7a44a",
        True,
    ),
]
TIERS = [
    (0, 9, "bottom"),
    (0, -11, "top"),
    (0, 21, "bottom"),
]  # cycled per cluster of labels


def event_label(name, info):
    """Short tick label for one EFD event."""
    if name == "MTAOS.closedLoopState":
        return AOS_LOOP_STATE.get(int(info["state"]), f"s{int(info['state'])}")
    if name == "ScriptQueue.script":
        return f"{int(info['scriptSalIndex']) % 1000}:{PROCESS_STATE.get(int(info['processState']), '?')}"
    if name == "Script.state":
        return f"{int(info['salIndex']) % 1000}:{SCRIPT_STATE.get(int(info['state']), '?')}"
    if name == "MTCamera.ccsCommandState":
        return f"ccs:{int(info['substate'])}"
    label = EVENT_LABEL.get(name, name)
    if "inPosition" in info:
        label += " T" if info["inPosition"] else " F"
    return label


def plot_gap_critical_path(pick_label, g, events, x_min=-1.5, x_max=None):
    """Two-panel critical-path figure for one gap row `g` with its EFD `events`.

    Pass a common `x_max` to put several gaps on directly comparable axes.
    """
    edges = np.concatenate([[0.0], np.cumsum([g[c] for c in SUBCOMPONENTS])])
    gap_len = g.visit_gap
    if x_max is None:
        x_max = gap_len + max(2.0, 0.10 * gap_len)
    x_span = x_max - x_min

    # Constrained layout instead of tight_layout: tight_layout is not compatible with
    # gridspec-built axes here and warns; constrained layout handles the labels cleanly.
    fig = plt.figure(figsize=(13, 11.5), layout="constrained")
    gs = fig.add_gridspec(2, 1, height_ratios=[1.45, 1.0])
    ax_top = fig.add_subplot(gs[0])
    ax_bot = fig.add_subplot(gs[1], sharex=ax_top)

    # --- TOP: the segment staircase (critical path) ---
    n_seg = len(SUBCOMPONENTS)
    for i, c in enumerate(SUBCOMPONENTS):
        y = n_seg - 1 - i  # first segment on the top row
        dur = g[c]
        _name, color = SEGMENTS[c]
        if dur > 0.005:
            ax_top.barh(
                y, dur, left=edges[i], height=0.62, color=color, edgecolor="k", lw=0.5
            )
            label = f"{dur:.2f} s"
            if (
                dur > 0.15 * x_span
            ):  # wide enough on the COMMON axis for the text to sit inside
                ax_top.text(
                    edges[i] + dur / 2, y, label, ha="center", va="center", fontsize=9
                )
            else:
                ax_top.text(
                    min(edges[i + 1] + 0.008 * x_span, x_max - 0.1),
                    y,
                    label,
                    ha="left",
                    va="center",
                    fontsize=9,
                    clip_on=True,
                )
        else:
            ax_top.plot(edges[i], y, "|", color="#999999", markersize=10)
            ax_top.text(
                edges[i] + 0.008 * x_span,
                y,
                "0 s",
                ha="left",
                va="center",
                fontsize=8,
                color="#888888",
            )
        if i < n_seg - 1:  # dotted hand-off connector down to the next segment
            ax_top.plot(
                [edges[i + 1]] * 2,
                [y - 0.31, y - 1 + 0.31],
                color="#666666",
                lw=0.9,
                ls=":",
            )
    ax_top.set_yticks(range(n_seg))
    ax_top.set_yticklabels(
        [SEGMENTS[c][0] for c in reversed(SUBCOMPONENTS)], fontsize=9
    )
    ax_top.set_ylim(-0.7, n_seg - 0.3)
    ax_top.tick_params(labelbottom=False)
    ax_top.grid(axis="x", alpha=0.3)
    ax_top.set_title(
        f"Deep dive, {pick_label}: gap before visit {g.visit_id} "
        f"({gap_len:.1f} s, {g.observation_reason}, band {g.prev_band}->{g.band})\n"
        "Top: mechanism segments tiling the gap (the critical path). Bottom: the raw EFD events.",
        loc="left",
        fontsize=10,
    )

    # --- BOTTOM: the raw events. Labels are thinned: at most 3 per cluster (one per tier),
    # where a new cluster starts when events are more than 2% of the axis span apart. The
    # printed dump above the figures remains the complete record.
    for y, (_row_label, names, color, label_events) in enumerate(ROW_STREAMS):
        ts_list = sorted(
            ((ts - g.t_close).total_seconds(), name, info)
            for ts, name, info in events
            if name in names
        )
        cluster_x, tier_i = None, 0
        for x, name, info in ts_list:
            ax_bot.plot(x, y, "|", markersize=15, color=color, markeredgewidth=2)
            if not label_events:
                continue
            if cluster_x is None or x - cluster_x >= 0.02 * x_span:
                cluster_x, tier_i = x, 0
            if tier_i >= len(TIERS):
                continue  # cluster already carries three labels; the dump has the rest
            dx, dy, va = TIERS[tier_i]
            ax_bot.annotate(
                event_label(name, info),
                (x, y),
                textcoords="offset points",
                xytext=(dx, dy),
                ha="center",
                va=va,
                fontsize=7.5,
                color=color,
                clip_on=True,
            )
            tier_i += 1
    ax_bot.set_yticks(range(len(ROW_STREAMS)))
    ax_bot.set_yticklabels([r[0] for r in ROW_STREAMS], fontsize=9)
    ax_bot.set_ylim(-0.8, len(ROW_STREAMS) - 0.2)
    ax_bot.set_xlabel("Seconds since shutter close of the previous visit")
    ax_bot.grid(axis="x", alpha=0.3)

    # Gap boundaries across both panels.
    for ax in (ax_top, ax_bot):
        ax.axvline(0, color="k", lw=1.0, alpha=0.6)
        ax.axvline(gap_len, color="k", lw=1.0, alpha=0.6)
    ax_top.set_xlim(x_min, x_max)


# One common axis for all the figures, set by the longest picked gap.
longest = max(g.visit_gap for g in deep_gaps.values())
common_x_max = longest + max(2.0, 0.10 * longest)
for pick_label, g in deep_gaps.items():
    plot_gap_critical_path(
        pick_label, g, deep_events_by_pick[pick_label], x_max=common_x_max
    )

## Full-night statistics

The per-visit time of each mechanism segment over all gaps in the analysis sample (every survey tier, strategy changes included): **median, mean, max and min** (the min over the gaps where the segment occurred, with how often it occurred), the night's total, and the share of the integrated gap time.

**Median and mean are reported separately** (the ticket asks for both): the median describes the *routine* visit, while the mean carries the long tail and is what multiplies into the integrated cost. A large mean/median ratio for a segment means its cost comes from occasional bad cases rather than from every visit; the max shows how bad the worst one was.

A table split by stratum and gap class follows, so the deep-drilling or strategy-change gaps can be compared or removed. **Mount slew+settle** is then broken down by its sub-category — which axis (azimuth or elevation) was last in position — with each axis's own motion time alongside. The AOS statistics are shown **conditional on each kind of correction being issued** — pre-slew, and post-slew with the in-position checks, correction, pointing correction and dispatch around it.

In [ ]:
# THE PER-SEGMENT TABLE (ticket deliverable): median and mean seconds per visit (plus min/max),
# over the analysis sample.
# Median, mean, total and share are over ALL gaps (a segment that did not occur counts as 0 s,
# which is what makes them add up to the gap). The MIN is over the gaps where the segment
# occurred (> PRESENT_S): several segments are legitimately 0 in many gaps -- the AOS segments
# and camera dispatch when there was no correction, and the rotator/dome catch-up when they
# finished inside the slew -- so a min over all gaps would just read 0. `present_in` says how
# often the segment occurred.
PRESENT_S = 1e-3  # a segment shorter than this counts as "did not occur"
sample = analysis_sample()


def component_stats(sample, cols):
    vals = sample[cols]
    present = vals > PRESENT_S
    tab = pd.DataFrame(
        {
            "median": vals.median(),
            "mean": vals.mean(),
            "min": vals.where(present).min(),
            "max": vals.max(),
            "present_in": present.mean(),
            "total_min_per_night": vals.sum() / 60,
        }
    )
    tab["share_of_gap_time"] = vals.sum() / sample.visit_gap.sum()
    return tab


print(
    f"Analysis sample: {len(sample)} gaps; by stratum {sample.stratum.value_counts().to_dict()}, "
    f"by class {sample.gap_class.value_counts().to_dict()}"
)

mech_tab = component_stats(sample, MECH_COMPONENTS + ["visit_gap"]).rename(
    index={"visit_gap": "TOTAL gap"}
)
print(
    "Mechanism segments (one owner per row). median / mean / min / max in seconds.\n"
    "  min excludes 0 s: it is taken only over the gaps where the segment occurred (present_in = fraction of gaps);\n"
    "  median, mean, total and share count a segment that did not occur as 0 s."
)
display(mech_tab.round(2))

# The same, per stratum, so the deep-drilling (and strategy-change) gaps can be split out.
print(
    "Median / mean gap (s) by stratum and class (the sample keeps them together; this is the split):"
)
display(
    sample.groupby(["stratum", "gap_class"])
    .visit_gap.agg(["size", "median", "mean"])
    .round(2)
)

# Mount slew+settle, by sub-category: which axis limited the slew (the last one in position).
# The rows split the Mount slew+settle segment exactly (c_slew_az + c_slew_el = c_slew_mount).
print(
    "\nMount slew+settle by limiting axis (sub-category of the mount segment; seconds):"
)
display(
    sample.groupby("slew_limited_by")
    .c_slew_mount.agg(
        n="size",
        median="median",
        mean="mean",
        min="min",
        max="max",
        total_min_per_night=lambda s: s.sum() / 60,
    )
    .round(2)
)
moved = sample[sample.t_motion.notna()]
print(
    f"  slew start latency (raDecTarget -> first axis moves): median {moved.c_slew_latency.median():.2f} s"
)
print(
    f"  each axis on its own (motion start -> that axis in position; overlapping, NOT additive): "
    f"azimuth median {moved.az_axis_s.median():.2f} s / mean {moved.az_axis_s.mean():.2f} s, "
    f"elevation median {moved.el_axis_s.median():.2f} s / mean {moved.el_axis_s.mean():.2f} s"
)
no_mount_motion = sample[sample.t_motion.isna()]
if len(no_mount_motion):
    print(
        f"  gaps where the mount did not move (no in-position transition; e.g. a rotator-only DDF move): "
        f"{len(no_mount_motion)} ({', '.join(str(v) for v in no_mount_motion.visit_id)})"
    )

# AOS, conditional on each kind of correction being issued.
pre = sample[sample.t_aos_pre.notna()]
post = sample[sample.t_post_flag.notna()]
none = sample[sample.t_aos_pre.isna() & sample.t_post_flag.isna()]
print(
    f"\nAOS Correction (pre-slew) on {len(pre)} of {len(sample)} gaps ({len(pre) / len(sample):.0%}): "
    f"median {pre.c_aos_pre.median():.2f} s, mean {pre.c_aos_pre.mean():.2f} s; "
    f"gap median {pre.visit_gap.median():.2f} s"
)
print(
    f"AOS Correction (post-slew) cycle on {len(post)} of {len(sample)} gaps ({len(post) / len(sample):.0%}); "
    f"gap median {post.visit_gap.median():.2f} s. Within those gaps (median / mean, s):"
)
for col in ["c_inpos", "c_aos_post", "c_aos_pointing", "c_dispatch"]:
    print(
        f"  {SEGMENT_NAME[col]:28s} {post[col].median():5.2f} / {post[col].mean():5.2f}"
    )
print(f"No AOS correction: gap median {none.visit_gap.median():.2f} s")

In [ ]:
# Distributions: one histogram per mechanism segment on a LOG count scale, so the tails are
# visible (on a linear scale the peak hides them). All panels share the same bins for
# comparability; values beyond the last bin are clipped into it (the panel title says how many).
HIST_MAX_S = 40.0
bins = np.arange(0, HIST_MAX_S + 0.5, 0.5)

fig, axes = plt.subplots(4, 4, figsize=(20, 15), sharex=True, sharey=True)
axes = axes.ravel()
for ax, comp in zip(axes, MECH_COMPONENTS):
    vals = sample[comp]
    n_over = int((vals > HIST_MAX_S).sum())
    ax.hist(
        vals.clip(upper=HIST_MAX_S), bins=bins, color=SEGMENT_COLOR[comp], alpha=0.85
    )
    ax.set_yscale("log")
    ax.axvline(
        vals.median(),
        color="k",
        linestyle="--",
        lw=1.2,
        label=f"median {vals.median():.1f} s",
    )
    ax.axvline(
        vals.mean(), color="k", linestyle=":", lw=1.2, label=f"mean {vals.mean():.1f} s"
    )
    ax.set_title(
        comp + (f"  ({n_over} above {HIST_MAX_S:.0f} s in last bin)" if n_over else ""),
        fontsize=11,
    )
    ax.legend(fontsize=9, loc="upper right")
    ax.grid(alpha=0.3, which="both")
    ax.set_xlabel("seconds")
    ax.set_ylabel("gaps (log scale)")
# One extra panel after the segments: the total gap, same axes, for reference; hide the rest.
ax = axes[len(MECH_COMPONENTS)]
vals = sample.visit_gap
n_over = int((vals > HIST_MAX_S).sum())
ax.hist(vals.clip(upper=HIST_MAX_S), bins=bins, color="#555555", alpha=0.85)
ax.set_yscale("log")
ax.axvline(
    vals.median(),
    color="k",
    linestyle="--",
    lw=1.2,
    label=f"median {vals.median():.1f} s",
)
ax.axvline(
    vals.mean(), color="k", linestyle=":", lw=1.2, label=f"mean {vals.mean():.1f} s"
)
ax.set_title(
    "TOTAL gap (for reference)"
    + (f"  ({n_over} above {HIST_MAX_S:.0f} s)" if n_over else ""),
    fontsize=11,
)
ax.legend(fontsize=9, loc="upper right")
ax.grid(alpha=0.3, which="both")
ax.set_xlabel("seconds")
for ax in axes[len(MECH_COMPONENTS) + 1 :]:
    ax.set_visible(False)

fig.suptitle(
    f"Mechanism-segment distributions over {len(sample)} gaps in the analysis sample, dayObs {day_obs} "
    "(log count scale; dashed = median, dotted = mean)",
    y=0.995,
    fontsize=12,
)
plt.tight_layout()

In [ ]:
# The whole night at a glance: each gap in the analysis sample as a stacked bar of its mechanism
# segments, in time order. Structure to look for: the pre-slew AOS correction segment, the
# occasional post-slew AOS cycle (correction + pointing correction + dispatch), dome/rotator
# spikes, long mount slews, and the steady orchestration floor.
plot_sample = sample.sort_values("obs_start_mjd")
x = np.arange(len(plot_sample))
bottom = np.zeros(len(plot_sample))

fig, ax = plt.subplots(figsize=(15, 4.5))
for comp in MECH_COMPONENTS:
    ax.bar(
        x,
        plot_sample[comp],
        bottom=bottom,
        width=1.0,
        color=SEGMENT_COLOR[comp],
        label=comp,
    )
    bottom += plot_sample[comp].values
ax.set_xlim(-1, len(plot_sample))
ax.set_ylim(0, min(60, bottom.max() * 1.05))
ax.set_xlabel("Gaps in the analysis sample, in time order through the night")
ax.set_ylabel("Seconds")
ax.legend(loc="upper right", ncol=7, fontsize=8)
ax.set_title(
    f"Every visit gap in the analysis sample of dayObs {day_obs}, decomposed into mechanism segments",
    loc="left",
)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

In [ ]:
# The set-aside samples, reported separately (not decomposed): filter changes and gaps after a
# non-survey image. (Deep drilling, twilight and strategy-change gaps are IN the analysis sample;
# the per-stratum split is in the statistics above.)
# Interruptions, fault-state gaps and gaps without shutter events are NOT part of this analysis
# at all: they are only identified and removed here; faults and interruptions are the subject of
# separate analyses (time-loss accounting / fault metrics). Only their removal is logged below.
def gap_summary(sample, label):
    if len(sample) == 0:
        print(f"{label:36s} none")
        return
    print(
        f"{label:36s} n={len(sample):4d}  median {sample.visit_gap.median():7.1f} s  "
        f"mean {sample.visit_gap.mean():7.1f} s  total {sample.visit_gap.sum() / 60:6.1f} min"
    )


in_strata = gaps[gaps.stratum.isin(SAMPLE_STRATA)]
print(f"Set-aside gap samples, dayObs {day_obs} (strata {SAMPLE_STRATA}):\n")
gap_summary(in_strata.query("gap_class == 'filter_change'"), "filter change")
gap_summary(
    in_strata.query("gap_class == 'after_non_survey'"), "after non-survey image"
)

excluded = ["interruption", "fault_state", "no_shutter_event"]
n_excl = gaps.gap_class.isin(excluded).sum()
print(
    f"\nExcluded from this analysis entirely (addressed in separate analyses): {int(n_excl)} gaps "
    f"({', '.join(f'{int((gaps.gap_class == c).sum())} {c}' for c in excluded)})"
)

# Filter changes: how long is the exchange itself inside those gaps?
fc = in_strata.query("gap_class == 'filter_change'").copy()
if len(fc) > 0:
    fc["t_fc_start"] = first_in_window(ev_filter_start, fc.t_close, fc.t_open)
    fc["t_fc_end"] = last_in_window(ev_filter_end, fc.t_close, fc.t_open)
    fc["filter_exchange_s"] = (fc.t_fc_end - fc.t_fc_start).dt.total_seconds()
    found = fc[fc.filter_exchange_s.notna() & (fc.filter_exchange_s > 0)]
    print(
        f"\nFilter-change gaps with start/endSetFilter found: {len(found)} of {len(fc)}"
    )
    if len(found) > 0:
        print(
            f"  exchange (startSetFilter -> endSetFilter): median {found.filter_exchange_s.median():.1f} s"
        )
        print(
            f"  rest of those gaps (handover + two slews):  median "
            f"{(found.visit_gap - found.filter_exchange_s).median():.1f} s"
        )

## Results

Two deliverables, then the caveats.

1. A **headline box**: the gap and, for each mechanism segment, its median, mean, min and max per visit, how often it occurred, and its share of the night's gap time.
2. The **ranked list of contributors by integrated cost**: the thirteen mechanism segments of the analysis sample plus whole filter-change gaps, ordered by total minutes on this night, and extrapolated to a **nights-per-year** cost. Each row has one owner. Interruptions and fault-state gaps are **not ranked** — they are excluded from this analysis and belong to separate analyses.

### How the nights-per-year extrapolation works (first pass)

The ticket asks for contributors ranked "by integrated nights-per-year cost". With one night analysed, the only honest version is a scaling: assume this night is typical of `EXTRAPOLATION_NIGHTS` observing nights per year, then

$$\text{nights-per-year cost} = \frac{\text{component minutes on this night} \times \texttt{EXTRAPOLATION\_NIGHTS}}{\text{length of this observable night in minutes}}$$

That is: *if this component's time could be removed entirely, how many extra observable nights per year would the survey get back?* Only a part of each component is actually removable (a slew has a physical minimum; readout sets a floor under the short components), so these numbers are **upper bounds on the recoverable time**, meant for ranking, not for budgeting. Multi-night aggregation replaces this extrapolation once the method runs over a range of nights.

### What is reported, and what is not yet

| Ticket item | Status |
|---|---|
| Per-component time per visit, median and mean separately | **Reported** (with min and max) for the thirteen mechanism segments; Mount slew+settle also by limiting axis (azimuth / elevation). |
| Deep-drilling / continuous-exposure sequences excluded or reported separately | **Tagged, kept in the sample**: every gap carries its stratum and class; the statistics show the split by stratum, and `SAMPLE_STRATA` / `SAMPLE_CLASSES` remove them when wanted. |
| Ranked list of loss contributors by integrated nights-per-year cost | **First pass below**, single-night extrapolation. Interruptions and fault-state gaps are out of scope (separate analyses). |
| Components vs a slew-model expectation (how much of each is *excess*?) | **Not yet.** Needs the `rubin_scheduler` kinematic model per visit, as in `FBS_Targets`. |
| AOS corrections that become ready after the slew started | **Separated**: *AOS Correction (post-slew)*, *AOS pointing correction* and the *Camera dispatch* after them, each bounded by its own event. |

In [ ]:
# HEADLINE: the routine visit gap and where it goes, segment by segment.
# Median / mean / share are over all gaps (absent segments count as 0 s, so they add up);
# min is over the gaps where the segment occurred (see PRESENT_S in the statistics cell).
# Two decimals, so sub-0.1 s segments (script startup, camera start) do not read as 0.
from IPython.display import HTML, display

no_aos = sample[sample.t_aos_pre.isna() & sample.t_post_flag.isna()]
TD = "<td style='text-align: right; padding: 2px 0 2px 14px;'>"


def stat_cells(vals):
    present = vals[vals > PRESENT_S]
    return (
        f"{TD}<b>{vals.median():.2f} s</b></td>{TD}{vals.mean():.2f} s</td>"
        f"{TD}{present.min() if len(present) else float('nan'):.2f} s</td>{TD}{vals.max():.2f} s</td>"
        f"{TD}{len(present) / len(vals):.0%}</td>"
    )


def rows_html(comps):
    return "".join(
        f"<tr><td style='padding: 2px 12px 2px 0;'>{c}</td>{stat_cells(sample[c])}"
        f"{TD}{sample[c].sum() / sample.visit_gap.sum():.0%}</td></tr>"
        for c in comps
    )


header_row = (
    "<tr><th style='text-align: left;'>&nbsp;</th>"
    + "".join(
        f"<th style='text-align: right; padding-left: 14px;'>{label}</th>"
        for label in ["median", "mean", "min", "max", "present in", "share"]
    )
    + "</tr>"
)
total_row = (
    f"<tr style='border-top: 1px solid #999;'><td style='padding: 2px 12px 2px 0;'><b>TOTAL gap</b></td>"
    f"{stat_cells(sample.visit_gap)}{TD}100%</td></tr>"
)
strata_txt = ", ".join(f"{n} {s}" for s, n in sample.stratum.value_counts().items())
az_share = (sample.slew_limited_by == "azimuth").mean()

display(
    HTML(
        f"""
<div style="border: 3px solid #2c5f8a; border-radius: 8px; padding: 14px 20px; margin: 10px 0; max-width: 980px;
            background: rgba(128, 128, 128, 0.08); font-family: sans-serif;">
  <div style="font-size: 1.05em; font-weight: bold; letter-spacing: 0.04em;">
    ROSP-23 INTER-VISIT INTERVAL BREAKDOWN &mdash; dayObs {day_obs}, {len(sample)} visit gaps ({strata_txt})
  </div>
  <div style="font-size: 2.6em; font-weight: bold; color: #2c5f8a; margin: 6px 0 0 0;">
    {sample.visit_gap.median():.1f} s median &nbsp;/&nbsp; {sample.visit_gap.mean():.1f} s mean
  </div>
  <div style="margin: 10px 0;">
    <div style="font-weight: bold; margin-bottom: 4px;">Mechanism segments, per visit gap (in time order)</div>
    <table style="border-collapse: collapse;">{header_row}{rows_html(MECH_COMPONENTS)}{total_row}</table>
  </div>
  <div style="font-weight: bold;">Notes</div>
  <ul style="margin: 2px 0 0 0;">
    <li>Gaps are timed from the EFD shutter events (endShutterClose &rarr; startShutterOpen);
        the segments sum exactly to each gap.</li>
    <li><b>min excludes 0 s</b>: it is taken only over the gaps where the segment occurred;
        <b>present in</b> is how often it occurred. Median, mean and share are over all gaps (a segment that
        did not occur counts as 0 s), which is why optional segments have a 0 s median. Segments that can be
        absent: the AOS segments and camera dispatch (no correction) and the rotator/dome catch-up (finished
        inside the slew).</li>
    <li>Mount slew+settle was limited by azimuth on {az_share:.0%} of gaps (the split by limiting axis is in the
        full-night statistics).</li>
    <li>AOS corrections were applied before the slew on {sample.t_aos_pre.notna().mean():.0%} of gaps
        and after it (with a pointing correction) on {sample.t_post_flag.notna().mean():.0%};
        gaps with neither have a median of {no_aos.visit_gap.median():.1f} s.</li>
    <li>Medians of segments do not sum to the median gap (medians are not additive); shares are of integrated time.</li>
    <li>The sample includes all survey tiers and strategy-change gaps (tagged, so they can be split out).
        Filter changes and gaps after non-survey images are reported separately;
        interruptions and fault-affected gaps are excluded from this analysis entirely (separate analyses).</li>
  </ul>
</div>
"""
    )
)

In [ ]:
# RANKED CONTRIBUTORS by integrated cost, with the first-pass nights-per-year extrapolation.
# The ranking uses the MECHANISM SEGMENTS of the analysis sample plus whole filter-change gaps.
# Interruptions and fault-state gaps are NOT ranked: they are excluded from this analysis and
# addressed in separate analyses (time-loss accounting / fault metrics).
EXTRAPOLATION_NIGHTS = (
    300  # assumed observing nights per year this night is typical of (first pass)
)

contributors = {
    c: sample[c].sum() / 60 for c in MECH_COMPONENTS
}  # minutes on this night
contributors["Filter changes (whole gaps)"] = (
    gaps[
        gaps.stratum.isin(SAMPLE_STRATA) & (gaps.gap_class == "filter_change")
    ].visit_gap.sum()
    / 60
)

ranked = pd.DataFrame({"minutes_per_night": pd.Series(contributors)}).sort_values(
    "minutes_per_night", ascending=False
)
night_minutes = night_hours * 60
ranked["share_of_observable_night"] = ranked.minutes_per_night / night_minutes
ranked["nights_per_year_cost"] = (
    ranked.minutes_per_night * EXTRAPOLATION_NIGHTS / night_minutes
)
ranked.index.name = "contributor"
display(ranked.round(2))

fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = [SEGMENT_COLOR.get(n, "#777777") for n in ranked.index[::-1]]
ax.barh(ranked.index[::-1], ranked.nights_per_year_cost[::-1], color=bar_colors)
for i, v in enumerate(ranked.nights_per_year_cost[::-1]):
    ax.text(v, i, f" {v:.1f}", va="center", fontsize=9, clip_on=True)
# Leave headroom on the right so the value labels stay inside the axes.
ax.set_xlim(0, ranked.nights_per_year_cost.max() * 1.12)
ax.set_xlabel(
    f"Nights-per-year cost (extrapolated from this night x {EXTRAPOLATION_NIGHTS} nights)"
)
ax.set_title(
    f"Ranked gap-time contributors (mechanism segments), dayObs {day_obs}\n"
    "(upper bounds on recoverable time; grey bar = whole filter-change gaps)",
    loc="left",
    fontsize=10,
)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()

print(
    f"\nContext: gap time in the analysis sample totals {sample.visit_gap.sum() / 60:.1f} min "
    f"of a {night_hours:.1f} h observable night "
    f"({sample.visit_gap.sum() / 3600 / night_hours:.1%})."
)

## Configuration-change markers

This section records **changes that can move the breakdown for reasons other than how the system performed**, so a step in the series is not mistaken for a trend. Two kinds, as in the other PTH notebooks:

- **Method changes**: different marker events, a different waterfall, a different selection. A change of definition is a **series break**, not a restatement. Current method: **v0.1 (draft)**.
- **System changes**: script or control-software changes that alter what the events mean. The code trace behind the markers pins the behaviour of specific repo versions; the planned script-synchronization improvements (moving the slew-flag set to coincide with shutter close, removing delay before TMA motion — see Lynne Jones's slides) would change the startup/prep/slew split *by design*, and must be recorded here when deployed. Changes in **how and when AOS corrections are applied** must be recorded too: e.g. applying a correction as soon as it is ready instead of after the in-position checks, not applying one that arrives after the slew before the exposure, or MTAOS's `enable_pointing_correction` setting (which decides whether a pointing offset is sent). **No marker depends on script text**: every boundary is a SAL command or event filtered only on enum-valued fields, so a script release that only rewords checkpoints cannot move the breakdown. (An early draft of v0.1 used the "Track target…" and "Take data." checkpoint strings as two sub-boundaries; the string dependence was removed before first use.)

### Current method decisions (v0.1 draft)

| Decision | Setting in this notebook | Status |
|---|---|---|
| Gap definition | EFD `MTCamera.endShutterClose`(N) to `startShutterOpen`(N+1); single-exposure visits (no snaps). ConsDB used only for visit identity and to pair shutter events to images (`SHUTTER_MATCH_TOL_S`). | Decided; supersedes the two-snap wording of OSS-REQ-0289. (An early draft used ConsDB `obs_end` / `obs_start`; the EFD gap is ~0.04 s longer.) |
| Marker assignment | By time window, first/last event in the gap; script markers restricted to main-queue salIndex (1xxxxx); **SAL commands/events and enum-valued fields only — never checkpoint strings** | Decided; cross-checked against script identity in the deep dive |
| Attribution | Serial waterfall of thirteen mechanism segments; parallel subsystems (rotator, dome) charged marginally beyond the mount | Decided for v0.1; alternatives (critical-path per moment) possible |
| Mount sub-category | Mount slew+settle is one segment; its split by the last axis in position (azimuth / elevation) is kept as a sub-category for exploration, not as two segments (an intermediate draft reported them as two segments, which showed a 0 for one axis in every gap) | Decided for v0.1 |
| Reporting | Mechanism segments only (median, mean, max, and min over the gaps where the segment occurred). An early draft also rolled them up into the ROSP-23 ticket's six categories; that comparison was dropped. | Decided for v0.1 |
| AOS corrections | **Pre-slew** (issued before the slew command) reported on its own. **Post-slew** (second slew-flag cycle) split at MTAOS `WAITING_IMAGE` into the correction and the **pointing correction** (to `clearSlewFlag`), followed by the **camera dispatch** (to `takeImages`). The checks before the cycle are **In-position checks**. | Decided for v0.1 |
| Post-slew / camera split | **In-position checks** (closed by the post-slew AOS cycle or `takeImages`) and, after a post-slew cycle, **Camera dispatch**, separated from **Camera start** (`takeImages` → `startShutterOpen`). Without a post-slew cycle the dispatch is not event-observable and stays inside In-position checks (the early draft used the "Take data." checkpoint for it and found ~0.1 s). | Decided for v0.1 |
| Analysis sample | `SAMPLE_STRATA` = main survey, deep drilling, twilight; `SAMPLE_CLASSES` = clean + strategy_change. Filter changes and gaps after non-survey images set aside. **No gap-length (outlier) cut.** | Decided for v0.1 (the early-draft 120 s `OUTLIER_GAP_S` cut was removed as not a valid filter) |
| Interruptions | Gap overlaps an `INTERRUPTION_STATES` = `IDLE` / `DOWNTIME` observatory-state period → excluded | Decided for v0.1 |
| Observatory-state guard | No `EXCLUDE_STATES = ["FAULT"]` period may overlap shutter open (N) → shutter close (N+1); neighbouring gaps also set aside (`EXCLUDE_NEIGHBOR_GAPS = True`) | Decided for v0.1 |
| Science programs | `SCIENCE_PROGRAMS` list from `FBS_Targets` (LSSTCam era) | Must track survey block changes |
| Extrapolation | `EXTRAPOLATION_NIGHTS = 300`, single-night scaling | Placeholder until multi-night aggregation |

In [ ]:
# Method changes and system changes that affect how the breakdown series should be read: {dayObs: description}.
# Add new entries (with their date) when a decision changes; a change of method is a series break.
METHOD_CHANGES = {}  # none yet: v0.1 is the first implementation
CONFIG_CHANGES = (
    {}
)  # record script/control-software deployments that change what the markers mean

for title, changes in [
    ("Method changes", METHOD_CHANGES),
    ("System changes", CONFIG_CHANGES),
]:
    print(f"{title}:")
    if changes:
        for dob, txt in sorted(changes.items()):
            print(f"  {dob}: {txt}")
    else:
        print("  none recorded")